In [1]:
from typing import List, Optional, Literal

import instructor
import pandas as pd

from google.genai import Client
from instructor import Mode
from instructor.v2 import from_genai
from pydantic import BaseModel, Field

PROJECT_ID = "leafy-guide-497515-m4"
LOCATION = "global"
MODEL_ID = "gemini-3.1-flash-lite"

In [2]:
raw_client = Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)

client = from_genai(
    raw_client,
    mode=Mode.JSON,
)

print(f"Project: {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Model: {MODEL_ID}")

Project: leafy-guide-497515-m4
Location: global
Model: gemini-3.1-flash-lite


In [3]:
class Response(BaseModel):
    name: str
    creator: str

In [22]:
response = client.chat.completions.create(
    model=MODEL_ID,
    response_model=Response,
    messages=[
        {"role": "user", "content": "Who are you and who did you create?"}
    ],
)

print(f'Name: {response.name}, Creator: {response.creator}')

Name: ChatGPT, Creator: OpenAI


In [9]:
df = pd.read_parquet("jobs.parquet")
df.head()

,raw_title,raw_location,company_name,salary_usd_min,salary_usd_max,level,raw_description
0,"Manager, Data Science (Machine Learning)","Johannesburg, South Africa",Standard Bank Group,48.0,90.0,Senior-level / Expert,View company page\n\nView company page\n\nAppl...
1,Mid-Level / Senior Data Engineer - Remote - La...,Peru,FullStack Labs,90.0,147.0,Mid-level / Intermediate,View company page\n\nView company page\n\nAppl...
2,Senior Marketing Data Scientist,"London, United Kingdom",Trainline,129.0,185.0,Senior-level / Expert,View company page\n\nView company page\n\nAppl...
3,Binance Accelerator Program - Data Scientist (...,"India, Mumbai",Binance,50.0,130.0,Entry-level / Junior,View company page\n\nView company page\n\nAppl...
4,US Master Data Manager,"New York City, United States",dentsu international,68.0,110.0,Senior-level / Expert,View company page\n\nView company page\n\nAppl...


In [10]:
df["raw_title"].sample(10).values

array(['Software Engineer, Data', 'Data Scientist (Hybrid)',
       'Applied Data Scientist',
       'Software (ML Product) Engineer (Staff/ Senior, Open Source, Python)',
       'Data Engineer | Customer Data Domain',
       'Applied Scientist, Intl Seller Growth',
       'Senior Product Data Analyst (Kraków, PL)', 'Data Engineer',
       'Machine Learning Engineer',
       'Staff Software Engineer, Infrastructure / Machine Learning'],
      dtype=object)

In [21]:
from tqdm import tqdm

In [23]:
def call_api(messages, response_model, model, max_retries=1):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
        response_model=response_model,
        max_retries=max_retries
    )

    return response

In [24]:
SeniorityLevel = Literal[
    "Intern",
    "Junior",
    "Mid-Level",
    "Senior",
    "Lead",
    "Staff",
    "Principal",
    "Manager",
    "Director",
    "Executive"
]

In [25]:
NormalizedTitle = Literal[
    "Data Scientist", "Data Engineer", "Data Analyst",
    "Analytics Engineer", "BI Developer",
    "Data Architect", "Solutions Architect",
    "Machine Learning Engineer", "MLOps Engineer",
    "AI Engineer", "Computer Vision Engineer", "NLP Engineer",
    "Research Scientist", "Applied Scientist",
    "Software Engineer", "Backend Engineer", "Fullstack Engineer",
    "DevOps Engineer", "Platform Engineer", "SRE",
    "Product Manager", "Product Owner",
    "Engineering Manager",
]

In [27]:
class JobTitleV5(BaseModel):
    seniority_level: Optional[SeniorityLevel] = Field(
        None,
        description="The level of experience required for the position."
    )

    normalized_title: Optional[NormalizedTitle] = Field(
        None,
        description=(
            "The normalized job title. "
            "Ignore seniority indicators such as Senior or Junior "
            "and any location information."
        )
    )

    is_management: bool = Field(
        False,
        description="""
        Does the position involve direct people management?
        True for: Manager, Director, VP, Head of.
        False for: Lead, Staff, Principal.
        These are technical leadership roles without HR responsibilities.
        """
    )

    is_research: bool = Field(
        False,
        description="""
        Is the position research-oriented?
        True for: Research Scientist, Research Engineer,
        Applied Scientist.
        These roles typically require publications or a Ph.D.
        False for: ML Engineer focused on production,
        Data Analyst.
        """
    )

    is_consulting: bool = Field(
        False,
        description="""
        Is the role consulting-oriented or client-facing?
        True for: Consultant, Solutions Architect,
        Pre-Sales, Customer Success.
        """
    )

    is_operations: bool = Field(
        False,
        description="""
        Is the role operations-oriented and focused on
        infrastructure or production systems?
        True for: MLOps, DevOps, DataOps, SRE,
        Platform Engineer.
        Typical focus areas include deployment,
        monitoring, and CI/CD.
        """
    )

    is_contractor: bool = Field(
        False,
        description="""
        Is the position contractual or temporary?
        True for: Contract, Contractor, Freelance,
        Fixed-Term Contract (FTC), B2B.
        """
    )


test_titles = [
    "Senior Machine Learning Engineer",  # is_management=False, is_operations=False
    "VP of Data Science",  # is_management=True
    "Contract Data Analyst - 6 months",  # is_contractor=True
    "MLOps Lead",  # is_operations=True
    "Research Scientist, NLP (Ph.D. required)",  # is_research=True
    "Solutions Architect - Financial Services",  # is_consulting=True
]

for title in test_titles:
    print(title)
    response = call_api(
        [{"role": "user", "content": title}],
        JobTitleV5,
        model=MODEL_ID,
        max_retries=2
    )
    print(response)
    print("=" * 100)

Senior Machine Learning Engineer
seniority_level='Senior' normalized_title='Machine Learning Engineer' is_management=False is_research=False is_consulting=False is_operations=False is_contractor=False
VP of Data Science
seniority_level='Executive' normalized_title='Data Scientist' is_management=True is_research=False is_consulting=False is_operations=False is_contractor=False
Contract Data Analyst - 6 months
seniority_level=None normalized_title='Data Analyst' is_management=False is_research=False is_consulting=False is_operations=False is_contractor=True
MLOps Lead
seniority_level='Lead' normalized_title='MLOps Engineer' is_management=False is_research=False is_consulting=False is_operations=True is_contractor=False
Research Scientist, NLP (Ph.D. required)
seniority_level=None normalized_title='NLP Engineer' is_management=False is_research=True is_consulting=False is_operations=False is_contractor=False
Solutions Architect - Financial Services
seniority_level=None normalized_title='So